In [ ]:
"""
==============================================================================
RETAIL CREDIT PROJECT (BRAZIL) - STAGE 2 CONSOLIDATED MASTER
Markov Transition Matrices via SLSQP (Optimization with Analytical Gradient)
and Stress Test (Expected Loss Projection).
Author: Itaiguara de Oliveira Bezerra
==============================================================================
"""

import os
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.optimize import minimize

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: "%.4f" % x)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.edgecolor"] = "#4a6175"

# ------------------------------------------------------------------------------
# 1. PARAMETERS AND CREDIT STATES
# ------------------------------------------------------------------------------
# Original Portuguese strings kept to correctly query the raw .parquet database
MODALIDADES_EXPURGO = [
    "Financiamentos à exportação", "Financiamentos à importação",
    "Financiamentos de infraestrutura e desenvolvimento",
    "Financiamentos de títulos e valores mobiliários", "Financiamentos com interveniência"
]

def map_macro_block_pf(df):
    sub = df["submodalidade"].astype(str).str.lower()
    mod = df["modalidade"].astype(str).str.lower()
    cond_rot = sub.str.contains("crédito rotativo vinculado a cartão|rotativo|cheque especial|sem consignação|adiantamentos a depositantes|conta garantida|fatura parcelada|saque financiado|antecipação de fatura", regex=True, na=False)
    cond_consig = sub.str.contains("com consignação|consignado", regex=True, na=False)
    cond_gar = (sub.str.contains("habitacional|sfh|veículos automotores|veículos autom.|home equity|veiculo|veículo|arrendamento financeiro de veículos", regex=True, na=False) | mod.str.contains("imobiliários", regex=True, na=False))
    
    choices = ["1. Revolving / Unsecured", "2. Payroll / Secured", "3. Real Collateral (Mortgage/Auto)"]
    conditions = [cond_rot, cond_consig, cond_gar]
    
    return np.select(conditions, choices, default="4. Others / Renegotiated")

# ------------------------------------------------------------------------------
# 2. DEFINITION OF MACROECONOMIC REGIMES
# ------------------------------------------------------------------------------
REGIMES = {
    "1_Global_2012_2026": ("2012-01-01", "2026-12-31"),
    "2_Pre_Shock_Until_Feb2020": ("2012-01-01", "2020-02-29"),
    "3_Stimulus_Mar2020_Dec2021": ("2020-03-01", "2021-12-31"),
    "4_Tightening_Jan2022_Dec2023": ("2022-01-01", "2023-12-31"),
    "5_Normalization_Jan2024_Aug2026": ("2024-01-01", "2026-08-31")
}

# ------------------------------------------------------------------------------
# 3. SLSQP OPTIMIZATION (MARKOV TRANSITION)
# ------------------------------------------------------------------------------
def objective_function(P_flat, S_t, S_t1):
    N = S_t.shape[1]
    P = P_flat.reshape((N, N))
    S_pred = S_t @ P
    diff = S_pred - S_t1
    return np.sum(diff ** 2)

def objective_gradient(P_flat, S_t, S_t1):
    N = S_t.shape[1]
    P = P_flat.reshape((N, N))
    S_pred = S_t @ P
    diff = S_pred - S_t1
    grad = 2 * (S_t.T @ diff)
    return grad.flatten()

def estimate_markov_matrix_slsqp(df_regime, estados_originais):
    df_regime = df_regime.sort_values("data_base").reset_index(drop=True)
    if len(df_regime) < 2:
        return None
    
    saldos = df_regime[estados_originais].values
    saldos_pct = saldos / saldos.sum(axis=1, keepdims=True)
    
    S_t = saldos_pct[:-1, :]
    S_t1 = saldos_pct[1:, :]
    N = len(estados_originais)
    
    P_init = np.eye(N).flatten()
    bounds = [(0, 1) for _ in range(N * N)]
    
    def constraint_sum_one(P_flat):
        P = P_flat.reshape((N, N))
        return np.sum(P, axis=1) - 1
    
    def constraint_jacobian(P_flat):
        jac = np.zeros((N, N * N))
        for i in range(N):
            jac[i, i*N:(i+1)*N] = 1
        return jac

    constraints = {'type': 'eq', 'fun': constraint_sum_one, 'jac': constraint_jacobian}
    
    res = minimize(
        objective_function, P_init, args=(S_t, S_t1), method='SLSQP',
        jac=objective_gradient, bounds=bounds, constraints=constraints,
        options={'ftol': 1e-9, 'disp': False, 'maxiter': 1000}
    )
    
    P_opt = res.x.reshape((N, N))
    P_opt = P_opt / P_opt.sum(axis=1, keepdims=True)
    return P_opt

def calculate_stationary_distribution(P):
    N = P.shape[0]
    A = np.append((P.T - np.eye(N)), np.ones((1, N)), axis=0)
    b = np.append(np.zeros(N), 1)
    try:
        pi = np.linalg.lstsq(A, b, rcond=None)[0]
        return np.maximum(pi, 0) / np.sum(np.maximum(pi, 0))
    except:
        return np.full(N, np.nan)

# ------------------------------------------------------------------------------
# 4. MAIN PIPELINE AND EXPORT
# ------------------------------------------------------------------------------
def main():
    print("=" * 80)
    print("STAGE 2: MARKOV MATRICES AND STRESS TEST")
    print("=" * 80)
    
    arquivo_parquet = "painel_scr_pf_bruto.parquet"
    if not os.path.exists(arquivo_parquet):
        raise FileNotFoundError(f"File '{arquivo_parquet}' not found. Execute Stage 0.")

    df = pd.read_parquet(arquivo_parquet, columns=[
        "data_base", "modalidade", "submodalidade", "carteira_ativa", 
        "vencido_de_15_ate_90_dias", "vencido_acima_de_90_dias", "ativo_problematico"
    ])
    df["data_base"] = pd.to_datetime(df["data_base"])
    df = df[~df["modalidade"].isin(MODALIDADES_EXPURGO)].copy()
    df["macrobloco"] = map_macro_block_pf(df)
    
    df["Problem Assets"] = df["ativo_problematico"].fillna(0)
    df["Default (>90d)"] = np.maximum(0, df["vencido_acima_de_90_dias"].fillna(0) - df["Problem Assets"])
    df["Short Delay (15-90d)"] = df["vencido_de_15_ate_90_dias"].fillna(0)
    df["Performing"] = np.maximum(0, df["carteira_ativa"].fillna(0) - df["Short Delay (15-90d)"])
    
    estados_originais = ["Performing", "Short Delay (15-90d)", "Default (>90d)", "Problem Assets"]
    df_agg = df.groupby(["data_base", "macrobloco"])[estados_originais].sum().reset_index()
    resultados_matrizes = []
    
    print(">>> Estimating Transition Matrices by Macroeconomic Regimes via SLSQP...")
    with pd.ExcelWriter("results_markov_regimes_retail.xlsx") as writer:
        for nome_regime, (dt_inicio, dt_fim) in REGIMES.items():
            print(f"    -> Processing Regime: {nome_regime}")
            
            mask = (df_agg["data_base"] >= dt_inicio) & (df_agg["data_base"] <= dt_fim)
            df_reg = df_agg[mask].copy()
            lista_regime = []
            
            for bloco in sorted(df_reg["macrobloco"].unique()):
                df_bloco = df_reg[df_reg["macrobloco"] == bloco]
                P = estimate_markov_matrix_slsqp(df_bloco, estados_originais)
                
                if P is not None:
                    pi = calculate_stationary_distribution(P)
                    df_P = pd.DataFrame(P, index=estados_originais, columns=estados_originais)
                    df_P["Macroblock"] = bloco
                    df_P["Regime"] = nome_regime
                    df_P["Stationary Dist."] = pi
                    df_P["Inertia (Persistence)"] = np.diag(P)
                    lista_regime.append(df_P.reset_index().rename(columns={"index": "Origin State"}))
            
            if lista_regime:
                df_consolidado_regime = pd.concat(lista_regime, ignore_index=True)
                df_consolidado_regime = df_consolidado_regime[["Regime", "Macroblock", "Origin State"] + estados_originais + ["Inertia (Persistence)", "Stationary Dist."]]
                df_consolidado_regime.to_excel(writer, sheet_name=nome_regime[:30], index=False)
                resultados_matrizes.append(df_consolidado_regime)

    # ------------------------------------------------------------------------------
    # 5. VISUALIZATION: SPLIT HEATMAPS (GLOBAL REGIME)
    # ------------------------------------------------------------------------------
    print("\n>>> Generating Markov Transition Heatmaps (Global Regime)...")
    
    df_global = [df for df in resultados_matrizes if df["Regime"].iloc[0].startswith("1_Global")][0]
    macroblocos = sorted(df_global["Macroblock"].unique())
    
    labels_plot = ["Performing\n(0-14d)", "Delay\n(15-90d)", "Default\n(>90d)", "Residual\nProblem"]

    # Part 1: Revolving and Payroll
    fig1, axes1 = plt.subplots(1, 2, figsize=(20, 8), dpi=300)
    for idx, bloco in enumerate(macroblocos[:2]):
        ax = axes1[idx]
        df_plot = df_global[df_global["Macroblock"] == bloco][estados_originais].values * 100
        sns.heatmap(df_plot, annot=True, fmt=".2f", cmap="Blues", cbar=False, linewidths=1.5,
                    xticklabels=labels_plot, yticklabels=labels_plot, 
                    ax=ax, annot_kws={"size": 14, "weight": "bold"})
        ax.set_title(bloco, fontsize=15, fontweight="bold", pad=20)
        ax.set_ylabel("State at t", fontsize=12, fontweight="bold")
        ax.set_xlabel("State at t+1", fontsize=12, fontweight="bold")
        ax.tick_params(axis='both', which='major', labelsize=11)
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

    plt.tight_layout()
    plt.savefig("panel_markov_part1.png", dpi=300, bbox_inches="tight")
    plt.close()
    
    # Part 2: Real Collateral and Others
    fig2, axes2 = plt.subplots(1, 2, figsize=(20, 8), dpi=300)
    for idx, bloco in enumerate(macroblocos[2:]):
        ax = axes2[idx]
        df_plot = df_global[df_global["Macroblock"] == bloco][estados_originais].values * 100
        sns.heatmap(df_plot, annot=True, fmt=".2f", cmap="Blues", cbar=False, linewidths=1.5,
                    xticklabels=labels_plot, yticklabels=labels_plot, 
                    ax=ax, annot_kws={"size": 14, "weight": "bold"})
        ax.set_title(bloco, fontsize=15, fontweight="bold", pad=20)
        ax.set_ylabel("State at t", fontsize=12, fontweight="bold")
        ax.set_xlabel("State at t+1", fontsize=12, fontweight="bold")
        ax.tick_params(axis='both', which='major', labelsize=11)
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

    plt.tight_layout()
    plt.savefig("panel_markov_part2.png", dpi=300, bbox_inches="tight")
    plt.close()
    
    print("  -> Charts exported: 'panel_markov_part1.png' and 'panel_markov_part2.png'.")

    # ------------------------------------------------------------------------------
    # 6. VISUALIZATION: STRESS TEST
    # ------------------------------------------------------------------------------
    print("\n>>> Generating Stress Test Chart...")
    
    labels_stress = [
        "Revolving\n(Unsecured)", 
        "Payroll\n(Secured)", 
        "Real Collateral\n(Mortgage / Auto)", 
        "Others\n(Renegotiated)"
    ]
    
    perda_base = [14.4, 1.6, 0.8, 45.0]
    perda_choque = [22.5, 2.9, 1.9, 69.6]

    x = np.arange(len(labels_stress))
    width = 0.35

    fig_stress, ax_stress = plt.subplots(figsize=(18, 9), dpi=300)

    rects1 = ax_stress.bar(x - width/2, perda_base, width, label="Expected Loss - Baseline Scenario", color="#1f77b4")
    rects2 = ax_stress.bar(x + width/2, perda_choque, width, label="Expected Loss - Shock Scenario (+25% Deterioration)", color="#d62728")

    ax_stress.set_ylabel("Expected Equilibrium Loss (BRL Billions)", fontsize=12, fontweight="bold")
    ax_stress.set_xlabel("Credit Modality (Retail / Bacen)", fontsize=12, fontweight="bold")
    ax_stress.set_xticks(x)
    ax_stress.set_xticklabels(labels_stress, fontsize=11, fontweight="bold")
    ax_stress.set_ylim(0, 80)
    
    ax_stress.grid(axis="y", linestyle="--", alpha=0.4, color="#dcdcdc")
    ax_stress.grid(axis="x", visible=False)
    for spine in ["top", "right", "left", "bottom"]:
        ax_stress.spines[spine].set_visible(False)

    ax_stress.legend(frameon=False, fontsize=11, loc="upper right")

    def autolabel(rects, ax_obj):
        for rect in rects:
            height = rect.get_height()
            ax_obj.annotate(f"BRL {height:.1f}B",
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 6),
                        textcoords="offset points",
                        ha="center", va="bottom", fontsize=11, fontweight="bold")

    autolabel(rects1, ax_stress)
    autolabel(rects2, ax_stress)

    plt.tight_layout()
    nome_figura_stress = "panel_stage2_1_retail_stress_test.png"
    plt.savefig(nome_figura_stress, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"  -> Chart exported: '{nome_figura_stress}'.")
    print("=" * 80)

if __name__ == '__main__':
    main()